In [21]:
import ee
import geemap
from utils import *

initialize()
config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

forests_included = 'allpixels'  # options: 'allpixels', 'edges_removed', 'edges_IPCC'

In [ ]:
age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020")
biomass = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').first().select("AGB").rename("biomass")
sd = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').first().select("SD").rename("sd")

biomass_downloaded = ee.Image(f"{data_folder}/esa_biomass")

# map = geemap.Map()
# # # map.addLayer(edge, {'min':0, 'max':1, 'palette':['red', 'blue']}, "edge")
# map.addLayer(age, {'min':0, 'max':30, 'palette':"viridis"}, "age")
# map.addLayer(biomass, {'min':0, 'max':300, 'palette':["blue", "red"]}, "biomass")
# map.addLayer(biomass_downloaded, {'min':0, 'max':300, 'palette':["blue", "red"]}, "biomass_downloaded")
# # map.addLayer(sd, {'opacity': 0.1}, "sd")
# map

In [9]:
export_image = age.addBands(biomass).addBands(sd)

grid = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary_allpixels")

def export_csv(image, n_chunks = 1, lu_name = None):

    properties_to_export = image.bandNames().getInfo()
    
    total_features = grid.size().getInfo()
    chunk_size = int(total_features * 1/n_chunks)

    def process_chunk(chunk_index):
        start = chunk_index * chunk_size
        chunk = grid.toList(chunk_size, start)
        selected_pixels = ee.FeatureCollection(chunk)

        unified_fc = image.reduceRegions(selected_pixels, ee.Reducer.first(), 30)

        task = ee.batch.Export.table.toDrive(
            collection = unified_fc,
            description = "esacci_sd_age",
            fileFormat = "CSV",
            selectors = [p for p in properties_to_export if p not in ['system:index', '.geo']]
        )
        task.start()

    for i in range(n_chunks):
        if i*chunk_size < total_features:
            process_chunk(i)


export_csv(export_image, n_chunks = 30)


# make polygons of the same age patches
# iteratively mask by age and keep only the pixels that have all valid neighbors (i.e. remove edges)


In [23]:
municipios = ee.FeatureCollection("projects/extents-490617/assets/municipios")
xingu = municipios.filter(ee.Filter.eq('NM_MUN', 'São Félix do Xingu'))

amazon = ee.FeatureCollection("projects/extents-490617/assets/biomes_br").filter(ee.Filter.eq('Bioma', 'Amazônia'))

age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020").clip(xingu)

vectors = age.reduceToVectors(
  geometry = amazon.geometry(),
  geometryType = 'polygon',
  scale = 30,
  eightConnected = True,
  maxPixels = 1e12,
  labelProperty = 'age'
)

# ee.batch.Export.table.toAsset(
#     collection = vectors,
#     description = "secondary_age_vectors",
#     assetId = f"{data_folder}/secondary_age_vectors"
# ).start()

map = geemap.Map()
map.addLayer(age, {'min':0, 'max':30, 'palette':"viridis"}, "age")
# # map.addLayer(vectors, {}, "vectors")
map



Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

EEException: Collection.first: Error in map(ID=+86032+53106):
Geometry.area: Unable to perform this geometry operation. Please specify a non-zero error margin.

In [29]:

vectors = age.reduceToVectors(
  geometry = map.draw_last_feature.geometry(),
  geometryType = 'polygon',
  scale = 30,
  eightConnected = True,
  maxPixels = 1e12,
  labelProperty = 'age'
)


# map.addLayer(vectors, {}, "vectors")

filteredPolygons = vectors.map(lambda feature: feature.set('area_m2', feature.geometry().area(maxError=1))).filter(ee.Filter.gt('area_m2', 10000))

largePolygons = vectors.filter(ee.Filter.gt('system:area', 10000))


map.addLayer(filteredPolygons, {}, "large polygons")